<a href="https://colab.research.google.com/github/sudokusus69/my-code-Portfolio/blob/main/2Makemore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
words = open('/content/drive/MyDrive/Colab Notebooks/names.txt', 'r').read().splitlines()

In [ ]:
words[:10]

In [ ]:
b = {}
for w in words:
        chs = ['<S>'] + list(w) + ['<E>']
        for ch1 , ch2 in zip(chs, chs[1:]):
                bigram = (ch1, ch2)
                b[bigram] = b.get(bigram, 0) + 1

In [ ]:
sorted(b.items(), key = lambda kv: -kv[1])

In [ ]:
import torch

In [ ]:
N = torch.zeros((27,27) , dtype= torch.int32)

chars = sorted(list(set(''.join(words))))
stoi =  {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos =  {i:s for s,i in stoi.items()}

In [ ]:

for w in words:
        chs = ['.'] + list(w) + ['.']
        for ch1 , ch2 in zip(chs, chs[1:]):
                ix1 = stoi[ch1]
                ix2 = stoi[ch2]
                N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt


%matplotlib inline
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [ ]:
p = N[0].float()
p = p / p.sum()
p

In [ ]:
g = torch.Generator().manual_seed(2147484)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[ix]

In [ ]:
g = torch.Generator().manual_seed(2147484)
p = torch.rand(3, generator=g)
p

In [ ]:
torch.multinomial(p, num_samples=200, replacement=True, generator=g)

In [ ]:
P = (N+1).float()
P /= P.sum(1, keepdim=True)


In [ ]:

for i in range(5):
    out = []
    ix = 0
    while True:

        p = P[ix]
        #p = N[ix].float()
        #p = p / p.sum()
        #p = torch.ones(27) / 27.0
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()

        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))

In [ ]:
from logging import log
log_likelihood = 0.0

n = 0
for w in ['yashfv']:
        chs = ['.'] + list(w) + ['.']
        for ch1 , ch2 in zip(chs, chs[1:]):
                ix1 = stoi[ch1]
                ix2 = stoi[ch2]
                prob = P[ix1, ix2]
                logprob = torch.log(prob)
                log_likelihood += logprob
                n += 1
                print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f} ')

print(f'{log_likelihood=}')
nll = -1 * log_likelihood
print(f'{nll=}')
print(f'{nll/n=}')

In [ ]:
# create the training set of bigrams (x,y)
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs

In [ ]:
ys

In [ ]:
# Randomly initialise 27 neuron's weights, each neuron gets 27 inputs...

g = torch.Generator().manual_seed(21478344)
W = torch.randn((27,27),generator = g, requires_grad=True)
xenc = f.one_hot(xs, num_classes=27).float()


In [ ]:
plt.imshow(xenc)

In [ ]:
import torch.nn.functional as f

#Forward Pass ...

xenc = f.one_hot(xs, num_classes=27).float()
logits = xenc @ W    # log-counts
counts = logits.exp()
probs = counts / counts.sum(1, keepdims = True)
loss = -probs[torch.arange(5), ys].log().mean()


In [ ]:
#Backwards Pass....

W.grad = None #gradient set to zero
loss.backward()

In [ ]:
print(loss.item())

In [ ]:
W.data += -1 * W.grad


# **New more organised and efficient model , built on all the code written above..**

In [ ]:
# create the dataset


xs, ys = [], []                             # Initialize empty lists to store inputs (xs) and target outputs (ys)

for w in words:                         # Loop through just the first word in the dataset (for demonstration)
    chs = (["."] + list(w) + ["."] )        # Add special start/end token '.' around the word (e.g., '.emma.')

    for ch1, ch2 in zip(
        chs, chs[1:]
    ):                           # Pair each character with the next character (e.g., ('.', 'e'), ('e', 'm'))
        ix1 = stoi[ch1]          # Convert input character ch1 to its integer index
        ix2 = stoi[ch2]          # Convert target character ch2 to its integer index
        xs.append(ix1)           # Add input index to xs list
        ys.append(ix2)           # Add target index to ys list

xs = torch.tensor(xs)               # Convert input list xs into a PyTorch tensor (1D vector)
ys = torch.tensor(ys)               # Convert target list ys into a PyTorch tensor (1D vector)
num = xs.nelement()                 # Get the total number of elements/examples in xs tensor
print("number of examples: ", num)  # Print how many bigram pairs were created

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)                  # Create a PyTorch random generator with a fixed seed for reproducible results
W = torch.randn((27, 27), generator=g, requires_grad=True)     # Initialize 27x27 weight matrix randomly, enabling gradient tracking for backprop

In [ ]:
# gradient descent
for k in range(100):             # Run the full training loop for 10 iterations/steps

    # forward pass
    xenc = f.one_hot(xs, num_classes=27).float()        # Convert input indices into one-hot encoded vectors (float type)
    logits = xenc @ W                                   # Multiply input vectors with weight matrix W to get raw output scores (log-counts)
    counts = logits.exp()                               # Exponentiate logits to get positive values (equivalent to counts)
    probs = counts / counts.sum(1, keepdims=True)       # Normalize counts along rows so each row sums to 1.0 (probabilities)
    loss = -probs[torch.arange(num), ys].log().mean()   # Calculate Negative Log-Likelihood Loss averaged across all examples
    + 0.01 * (W**2).mean()                              # Regularization term added to the loss: it pushes weights towards zero when they aren't needed

    print(loss.item())                                  # Print the loss value as a standard Python number


    # backward pass
    W.grad = None                 # Reset gradients to None so they don't accumulate from previous steps
    loss.backward()               # Run backpropagation to compute the gradients of loss with respect to W (saved in W.grad)

    # update
    W.data += -20 * W.grad       # Tweak weight values slightly in the opposite direction of gradients using a step size (learning rate) of 0.1

In [155]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)  # Create a generator with fixed seed for reproducible sampling

for i in range(5):  # Generate 5 sample names

    out = []  # List to hold the characters of the current generated name
    ix = 0  # Start with character index 0 (the '.' start token)

    while True:

        # ----------
        # BEFORE:
        # p = P[ix]
        # ----------
        # NOW:
        xenc = f.one_hot(torch.tensor([ix]), num_classes=27).float()  # One-hot encode the single current character index
        logits = xenc @ W  # Compute raw output scores (predict log-counts)
        counts = logits.exp()  # Exponentiate logits to get counts (equivalent to N)
        p = counts / counts.sum(1, keepdims=True)  # Convert counts into probabilities for the next character
        # ----------

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # Sample next character index based on probabilities
        out.append(itos[ix])  # Convert index back to character string and append to output list

        if ix == 0:  # If the sampled character is the end token '.', stop generating this word
            break

    print(''.join(out))  # Join character list into a string and print the generated name

cexze.
momasurailezityha.
konimittain.
llayn.
ka.
